# Week 2, Day 5: A WrestleMania guide that speaks and draws

**Lesson:** [week2/day5.ipynb](../../../week2/day5.ipynb), *Airline AI assistant: going multi-modal*. **About 40 minutes.**

**What you practice here**

- Generating an image with `client.images.generate`, and decoding the base64 picture it sends back.
- Generating speech with `client.audio.speech.create`, and taking the audio's bytes from the response.
- One agent turn that uses three models: the tool loop finds what the user asked about, then the reply is spoken and the event drawn.
- The `gr.Blocks` callbacks: `put_message_in_chatbot`, then `chat(history)` returning the chatbot, the audio and the image.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week2/day5", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson's assistant uses three models in one turn. The chat model answers, calling the price tool when it needs a
price; `talker` reads the answer aloud with a text-to-speech model; and `artist` paints the first city the tool was
asked about, with an image model. A `gr.Blocks` page shows the chat, plays the audio and shows the picture. Build the
same for a WrestleMania guide: it speaks every answer, and draws a poster of the event that a tool looked up.

Day 4's date tool is given below, ready to run. The functions that call a model, `artist`, `talker` and `chat`, take
the client as their first parameter and make the calls themselves: an image generation, a speech generation and Chat
Completions. The checks hand them a stand-in shaped like the real client, which sends back a tiny real PNG and a few
bytes of audio, so nothing is spent. The `gr.Blocks` page itself goes in Try it for real.

## What each call should give

```python
artist(client, "WrestleMania III")
# makes one image request with the model gpt-image-1-mini, for one 1024x1024 picture,
# and returns the picture's PNG bytes: b'\x89PNG\r\n...'

talker(client, "WrestleMania III was held on 29 March 1987.")
# makes one speech request with the model gpt-4o-mini-tts and the voice onyx, and returns the audio's bytes

put_message_in_chatbot("When was WrestleMania III?", [])
# ('', [{'role': 'user', 'content': 'When was WrestleMania III?'}])

history, voice, image = chat(client, [{"role": "user", "content": "When was WrestleMania III?"}])
# history ends with {'role': 'assistant', 'content': 'WrestleMania III was held on 29 March 1987.'},
# voice is that reply read aloud, and image is a poster of WrestleMania III (None when no tool was called)
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import base64
import json

## Given

Day 4's date tool, ready to run, so that this notebook stands alone: the system message `SYSTEM`, the tool
`get_event_date`, and `TOOLS`, the list you pass as the tools of a Chat Completions call. Run this cell: your `chat`
uses all three.

In [ ]:
SYSTEM = """You are a WrestleMania guide. Give short, friendly answers of one or two sentences.
Use get_event_date for any date. If you don't know the answer, say so."""

DATES = {"wrestlemania i": "31 March 1985", "wrestlemania iii": "29 March 1987",
         "wrestlemania x": "20 March 1994", "wrestlemania 32": "3 April 2016"}


def get_event_date(event):
    return DATES.get(event.strip().lower(), "unknown")


TOOLS = [{"type": "function", "function": {
    "name": "get_event_date",
    "description": "Get the date of a WrestleMania, such as WrestleMania III. Use it whenever you are asked a date.",
    "parameters": {"type": "object",
                   "properties": {"event": {"type": "string", "description": "The event's name, such as WrestleMania III"}},
                   "required": ["event"], "additionalProperties": False}}}]

### 1 · `artist(client, subject, model="gpt-image-1-mini")`

Draws a poster of a subject, and returns the picture as bytes. Make one image generation request with the model
`model`, for one picture (`n=1`) of size `"1024x1024"`, with a prompt of your own that names `subject`; the lesson asks
for a vibrant pop-art style. The picture comes back as base64 text in the response's first item: decode it, and
return the PNG's bytes. The lesson then opens them with PIL's `Image.open(BytesIO(...))`; the checks have no PIL, so
that step goes in Try it for real.

<details><summary>Hint</summary>

```python
response = client.images.generate(model=model, prompt=prompt, size="1024x1024", n=1)
return base64.b64decode(response.data[0].b64_json)
```

</details>

In [ ]:
def artist(client, subject, model="gpt-image-1-mini"):
    # Write artist here, as described above.
    raise NotImplementedError("artist is not written yet")

In [ ]:
check("artist")

### 2 · `talker(client, text, model="gpt-4o-mini-tts", voice="onyx")`

Reads a text aloud, and returns the audio as bytes. Make one speech generation request with the model `model`, the
voice `voice` and `text` as its input, and return the response's content, the audio's bytes, as the lesson's `talker`
does.

<details><summary>Hint</summary>

```python
return client.audio.speech.create(model=model, voice=voice, input=text).content
```

</details>

In [ ]:
def talker(client, text, model="gpt-4o-mini-tts", voice="onyx"):
    # Write talker here, as described above.
    raise NotImplementedError("talker is not written yet")

In [ ]:
check("talker")

### 3 · `handle_tool_calls_and_return_events(message)`

Runs the date tool for each call the model made, and notes which events were asked about. It is the lesson's
`handle_tool_calls_and_return_cities`, for events. `message` is the model's reply as the SDK gives it. For each of its
tool calls to `get_event_date`, read the event from the call's arguments (JSON text), add a tool message that holds
`get_event_date(event)` and the call's id to the answers, and add the event, as the model wrote it, to the events.
(`TOOLS` holds only this tool; if you add another, answer its calls too, as on Day 4.) Return `(answers, events)`,
both in the calls' order.

<details><summary>Hint</summary>

```python
for call in message.tool_calls:
    if call.function.name == "get_event_date":
        event = json.loads(call.function.arguments)["event"]
        ...  # {"role": "tool", "content": get_event_date(event), "tool_call_id": call.id}
```

</details>

In [ ]:
def handle_tool_calls_and_return_events(message):
    # Write handle_tool_calls_and_return_events here, as described above.
    raise NotImplementedError("handle_tool_calls_and_return_events is not written yet")

In [ ]:
check("handle_tool_calls_and_return_events")

### 4 · `put_message_in_chatbot(message, history)`

The page's first callback: it shows the user's message in the chat at once, and clears the textbox. Return two
values: `""`, which Gradio puts in the textbox, and a new list for the chatbot, holding the history's entries, then a
user message holding `message`. This is the lesson's `put_message_in_chatbot`. Do not change the history you were
given.

<details><summary>Hint</summary>

`return "", history + [{"role": "user", "content": message}]`

</details>

In [ ]:
def put_message_in_chatbot(message, history):
    # Write put_message_in_chatbot here, as described above.
    raise NotImplementedError("put_message_in_chatbot is not written yet")

In [ ]:
check("put_message_in_chatbot")

### 5 · `chat(client, history, model="gpt-4.1-mini")`

The page's second callback: one turn that answers, speaks and draws. The history already ends with the user's
message, which `put_message_in_chatbot` added. Build a system message holding `SYSTEM`, then the history with only
`role` and `content`, and run Day 4's loop with `TOOLS` as the tools: while the reply asks for tools, add its message,
then the answers from `handle_tool_calls_and_return_events`, add the events it returns to a list, `events`, that
starts empty, and ask again. Then, from the reply's text: speak it with `talker`, and draw `events[0]` with `artist` if
a tool was called (with no tool call, the image is `None`). Return `(new_history, voice, image)`, where `new_history`
is the history with only `role` and `content`, then an assistant message holding the reply. Leave the history you
were given unchanged.

<details><summary>Hint</summary>

The lesson's `chat(history)`, with `history` stripped first, `events += found` in the loop, and at the end:

```python
voice = talker(client, reply)
image = artist(client, events[0]) if events else None
return history + [{"role": "assistant", "content": reply}], voice, image
```

</details>

In [ ]:
def chat(client, history, model="gpt-4.1-mini"):
    # Write chat here, as described above.
    raise NotImplementedError("chat is not written yet")

In [ ]:
check("chat")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 2, Day 5 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Each picture from `gpt-image-1-mini` costs about 3 cents, as the lesson warns; the speech and the answers cost a
fraction of a cent. Mind how many pictures you make.

1. Load your key with `load_dotenv(override=True)`, make `openai = OpenAI()`, and look at one picture:
   `Image.open(BytesIO(artist(openai, "WrestleMania III")))`, with `from PIL import Image` and `from io import BytesIO`.
2. Write `respond(history)`, which calls `chat(openai, history)` and turns the image's bytes into a PIL image (or
   `None`) for `gr.Image`.
3. Build the lesson's `gr.Blocks` page, after `import gradio as gr`: a row with `gr.Chatbot(height=500,
   type="messages")` and `gr.Image(height=500, interactive=False)`, a row with `gr.Audio(autoplay=True)`, and a row
   with a `gr.Textbox` named `message`.
4. Chain the callbacks: `message.submit(put_message_in_chatbot, ...)` with the textbox and the chatbot as inputs and
   outputs, then `.then(respond, ...)` from the chatbot to the chatbot, the audio and the image. Launch the page and
   ask "When was WrestleMania III?".
5. Optional, the lesson's exercise: add a second tool, such as `book_seats(event, seats)`, describe it in `TOOLS`,
   and answer its calls in `handle_tool_calls_and_return_events`. Then ask the guide to book two seats.

<details><summary>Stuck? One way to write it</summary>

```python
from io import BytesIO

from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
from PIL import Image

load_dotenv(override=True)
openai = OpenAI()
display(Image.open(BytesIO(artist(openai, "WrestleMania III at the Pontiac Silverdome"))))  # about 3 cents
```

```python
import gradio as gr


def respond(history):
    history, voice, image = chat(openai, history)
    return history, voice, Image.open(BytesIO(image)) if image else None


with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Ask about any WrestleMania:")
    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        respond, inputs=chatbot, outputs=[chatbot, audio_output, image_output])

ui.launch(inbrowser=True)
```

```python
# Optional: a second tool, as the lesson's exercise suggests
BOOKINGS = {}


def book_seats(event, seats):
    BOOKINGS[event] = BOOKINGS.get(event, 0) + seats
    return f"Booked {seats} seats for {event}, {BOOKINGS[event]} in all"


TOOLS.append({"type": "function", "function": {
    "name": "book_seats",
    "description": "Book seats at a WrestleMania for the fan.",
    "parameters": {"type": "object",
                   "properties": {"event": {"type": "string", "description": "The event's name"},
                                  "seats": {"type": "integer", "description": "How many seats to book"}},
                   "required": ["event", "seats"], "additionalProperties": False}}})


def handle_tool_calls_and_return_events(message):
    answers, events = [], []
    for call in message.tool_calls:
        arguments = json.loads(call.function.arguments)
        if call.function.name == "get_event_date":
            events.append(arguments["event"])
            result = get_event_date(arguments["event"])
        elif call.function.name == "book_seats":
            result = book_seats(arguments["event"], arguments["seats"])
        else:
            result = "unknown tool"
        answers.append({"role": "tool", "content": result, "tool_call_id": call.id})
    return answers, events


history, voice, image = chat(openai, [{"role": "user", "content": "Book me two seats for WrestleMania 32, please."}])
print(history[-1]["content"], BOOKINGS)
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.